# 05 · Sky-vs-Structure Segmentation

**Goal:** segment each frame into "sky" and "structure" two ways and
compare them as an ablation.

- **Classical** (FROM SCRATCH, `src/segmentation.py::classical_sky_mask`):
  brightness threshold + Canny edges + morphological closing.
- **SAM** (library-wrapped, `src/segmentation.py::sam_sky_mask`):
  Segment Anything with a trunk-base point prompt.

Both functions share signature `(image_bgr, **kwargs) -> bool mask`,
with the convention `mask == True` → structure. That makes swapping
them in notebook 06 (the reprojection filter) a one-line change.

In [ ]:
# Standard preamble — every notebook seeds np.random.seed(131).
import sys
from pathlib import Path

# Make `src/` importable from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
np.random.seed(131)

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src import segmentation, viz
    import cv2

    TREE_ID = "tree_oak_01"
    FRAMES_DIR = PROJECT_ROOT / "data" / "frames" / TREE_ID


if not FRAMES_DIR.exists():
    raise FileNotFoundError(
        f"Frames directory not found at {FRAMES_DIR}.\n"
        "Run notebook 01 to extract frames for this tree."
    )


    frame_paths = sorted(FRAMES_DIR.glob("frame_*.png"))
    sample_path = frame_paths[len(frame_paths) // 2]
    sample = cv2.imread(str(sample_path))
    print(f"Sample frame: {sample_path.name}  shape={sample.shape}")

## 1. Classical mask — from scratch

In [ ]:
mask_classical = segmentation.classical_sky_mask(sample)
print(f"Classical: {mask_classical.mean()*100:.1f}% of pixels are structure")

## 2. SAM mask — library

First run downloads/loads the ViT-B checkpoint. Subsequent calls reuse
the in-memory predictor.

In [ ]:
sam = segmentation.SamSegmenter()    # raises with a friendly message if checkpoint missing
mask_sam = sam(sample)
print(f"SAM:       {mask_sam.mean()*100:.1f}% of pixels are structure")

## 3. Side-by-side comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
viz.show_image(sample, title="Frame", ax=axes[0])
viz.plot_mask_overlay(sample, mask_classical, title="Classical (from scratch)", ax=axes[1])
viz.plot_mask_overlay(sample, mask_sam, title="SAM (library)", ax=axes[2], color=(0, 200, 255))
fig.suptitle(f"{TREE_ID} — segmentation comparison (IoU={segmentation.mask_iou(mask_classical, mask_sam):.3f})")
viz.save_fig(fig, f"05_{TREE_ID}_segmentation_comparison.png")
plt.show()

## 4. Batch-process every frame for the filter stage

Notebook 06 needs a structure mask per frame; cache them on disk so we
don't re-run SAM every time we tune the filter.

In [ ]:
MASKS_DIR = PROJECT_ROOT / "outputs" / "reconstructions" / f"{TREE_ID}_masks"
for variant_name, fn in [("classical", segmentation.classical_sky_mask), ("sam", sam)]:
    out = MASKS_DIR / variant_name
    out.mkdir(parents=True, exist_ok=True)
    for p in frame_paths:
        img = cv2.imread(str(p))
        m = fn(img)
        segmentation.save_mask(m, out / (p.stem + ".png"))
    print(f"Wrote {len(frame_paths)} masks → {out}")